In [10]:
# Imports

import dspy
import mlflow
import os

from dspy.datasets import MATH
from dotenv import load_dotenv

In [12]:
# Load API key from .env

load_dotenv()
if not os.getenv('OPENAI_API_KEY'):
    print("OPENAI_API_KEY or .env do not exist...")
    api_key = userdata.get('OPENAI_API_KEY')
    print("Using userdata's OPENAI_API_KEY")
else:
    api_key = os.getenv('OPENAI_API_KEY')
    print("Got OPENAI_API_KEY from .env")

Got OPENAI_API_KEY from .env


In [15]:
# Set up mlflow

# In seperate terminal on machine: mlflow ui --port 5000

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("DSPy")
mlflow.dspy.autolog()

2026/05/24 10:33:32 INFO mlflow.tracking.fluent: Experiment with name 'DSPy' does not exist. Creating a new experiment.


Trace(trace_id=tr-478e2cf3ca3492fbfd1f375857da2ca6)

In [3]:
# Set up LM

gpt4o_mini = dspy.LM('openai/gpt-4o-mini', max_tokens=2000) # Prompt LM
gpt4o = dspy.LM('openai/gpt-4o', max_tokens=2000) # Optimizer LM
dspy.configure(lm=gpt4o_mini)  # we'll use gpt-4o-mini as the default LM, unless otherwise specified

In [4]:
# Set up math dataset

dataset = MATH(subset='algebra')
print(len(dataset.train), len(dataset.dev))

README.md: 0.00B [00:00, ?B/s]

algebra/train-00000-of-00001.parquet:   0%|          | 0.00/505k [00:00<?, ?B/s]

algebra/test-00000-of-00001.parquet:   0%|          | 0.00/353k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1744 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1187 [00:00<?, ? examples/s]

350 350


In [5]:
# Inspect example

example = dataset.train[0]
print("Q: ", example.question)
print("A: ", example.answer)

Q:  The doctor has told Cal O'Ree that during his ten weeks of working out at the gym, he can expect each week's weight loss to be $1\%$ of his weight at the end of the previous week. His weight at the beginning of the workouts is $244$ pounds. How many pounds does he expect to weigh at the end of the ten weeks? Express your answer to the nearest whole number.
A:  221


In [13]:
# Set up module

module = dspy.ChainOfThought("question -> answer")
module(question=example.question)

Prediction(
    reasoning="Cal O'Ree's weight loss each week is given as \\(1\\%\\) of his weight at the end of the previous week. Therefore, if his weight at the beginning is \\(W_0 = 244\\) pounds, the weight at the end of week \\(n\\) can be expressed as:\n\n\\[\nW_n = W_{n-1} - 0.01 W_{n-1} = W_{n-1} (1 - 0.01) = W_{n-1} \\times 0.99\n\\]\n\nThis implies that:\n\n\\[\nW_n = W_0 \\times 0.99^n\n\\]\n\nTo find his weight after ten weeks, we substitute \\(n = 10\\) into the equation:\n\n\\[\nW_{10} = 244 \\times 0.99^{10}\n\\]\n\nCalculating \\(0.99^{10}\\):\n\n\\[\n0.99^{10} \\approx 0.904382\n\\]\n\nNow, we compute:\n\n\\[\nW_{10} \\approx 244 \\times 0.904382 \\approx 220.377368\n\\]\n\nRounding this to the nearest whole number:\n\n\\[\nW_{10} \\approx 220\n\\]\n\nThus, at the end of ten weeks, Cal O'Ree can expect to weigh approximately 220 pounds.",
    answer='220'
)

In [16]:
# Set up evaluator for zero-shot module above before optimization

THREADS = 24
kwargs = dict(num_threads=THREADS, display_progress=True, display_table=5)
evaluate = dspy.Evaluate(devset=dataset.dev, metric=dataset.metric, **kwargs)

evaluate(module)

Average Metric: 247.00 / 350 (70.6%): 100%|███████████████████████████████████████████| 350/350 [00:04<00:00, 79.23it/s]

2026/05/24 10:34:09 INFO dspy.evaluate.evaluate: Average Metric: 247 / 350 (70.6%)


,question,example_reasoning,example_answer,pred_reasoning,pred_answer,_patched
0,What is the smallest integer value of $c$ such that the function $...,The given function has a domain of all real numbers if and only if...,1,For the function \( f(x) = \frac{x^2 + 1}{x^2 - x + c} \) to have ...,1,✔️ [True]
1,What is the least value of $x$ that is a solution of $|{-x+3}|=7$?,"In order to have $|{-x+3}| = 7$, we must have $-x + 3 = 7$ or $-x ...",-4,"To solve the equation \(|{-x + 3}|=7\), we start by considering th...",-4,✔️ [True]
2,Evaluate $\left\lceil -\frac{7}{4}\right\rceil$.,"$-\frac{7}{4}$ is between $-1$ and $-2$, so $\left\lceil -\frac{7}...",-1,"To evaluate \(\left\lceil -\frac{7}{4}\right\rceil\), we first nee...",-1,✔️ [True]
3,"A triangle has vertices at coordinates $(11,1)$, $(2,3)$ and $(3,7...",We must find the distance between each pair of points by using the...,10,To find the longest side of the triangle with vertices at points \...,10,✔️ [True]
4,Let $f(x) = x + 2$ and $g(x) = 1/f(x)$. What is $g(f(-3))$?,"First, we find that $f(-3) = (-3) + 2 = -1$. Then, $$g(f(-3)) = g(...",1,"To find \( g(f(-3)) \), we first need to evaluate \( f(-3) \). The...",1,✔️ [True]


EvaluationResult(score=70.57, results=<list of 350 results>)

[Trace(trace_id=tr-b9c5d7fcdf2dbf8b9bdfd5f7b084bb73), Trace(trace_id=tr-6157746e2f73ab4c9d30097d0f693245), Trace(trace_id=tr-6bbd4d8257daa0f20dee41fa4da0dbf5), Trace(trace_id=tr-73aea28c46d29edeefeb6da575529612), Trace(trace_id=tr-bd28cbb15d8d307aefed5a10ac8260e8), Trace(trace_id=tr-610456e9f42f69a1cfdbeff34584c448), Trace(trace_id=tr-eaaafab67f6e0ba732fb5758426c344b), Trace(trace_id=tr-0f57a51fc65803ae9695203f7838d745), Trace(trace_id=tr-8627023dcbafb4d75778c560c2881d18), Trace(trace_id=tr-9c9a2903b309455eb5a00d698a1b4dff)]

In [17]:
# Track evaluation with mlflow

# Start an MLflow Run to record the evaluation
with mlflow.start_run(run_name="math_evaluation"):
    kwargs = dict(num_threads=THREADS, display_progress=True)
    evaluate = dspy.Evaluate(devset=dataset.dev, metric=dataset.metric, **kwargs)

    # Evaluate the program as usual
    result = evaluate(module)

    # Log the aggregated score
    mlflow.log_metric("correctness", result.score)
    # Log the detailed evaluation results as a table
    mlflow.log_table(
        {
            "Question": [example.question for example in dataset.dev],
            "Gold Answer": [example.answer for example in dataset.dev],
            "Predicted Answer": [output[1] for output in result.results],
            "Correctness": [output[2] for output in result.results],
        },
        artifact_file="eval_results.json",
    )

Average Metric: 11.00 / 14 (78.6%):   4%|█▋                                            | 13/350 [00:00<00:09, 35.26it/s]

2026/05/24 10:35:33 WARNING mlflow.tracing.export.mlflow_v3: Failed to send trace to MLflow backend: BAD_REQUEST: (sqlite3.IntegrityError) NOT NULL constraint failed: assessments.trace_id
[SQL: UPDATE assessments SET trace_id=? WHERE assessments.assessment_id = ?]
[parameters: (None, 'a-d85bf8896c3349c49cf3c207055a41bb')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


Average Metric: 247.00 / 350 (70.6%): 100%|███████████████████████████████████████████| 350/350 [00:06<00:00, 57.37it/s]

2026/05/24 10:35:39 INFO dspy.evaluate.evaluate: Average Metric: 247 / 350 (70.6%)



🏃 View run math_evaluation at: http://localhost:5000/#/experiments/1/runs/8fdde9afe0bc4dfcb72301dffd0a5b4a
🧪 View experiment at: http://localhost:5000/#/experiments/1


[Trace(trace_id=tr-4316269955d31a9757e4536add77f239), Trace(trace_id=tr-a27e34cc3b131c6ea23723700eccd54f), Trace(trace_id=tr-51e0e4f102498d6e2480dc81024adca0), Trace(trace_id=tr-27135a737b0811212a57ac036545c9e5), Trace(trace_id=tr-12d983cf8cd7a8bce94637cd4ab983f1), Trace(trace_id=tr-258b5982bed7396e8e7400fa3504aa22), Trace(trace_id=tr-05f04a40a7a7ec25cd5022102b0569dc), Trace(trace_id=tr-75e451954339edd23f3ef444a047278b), Trace(trace_id=tr-749a41500339a8639c492ff5f072ca90), Trace(trace_id=tr-e19ef359ae0a12436a1dbb3214dca557)]

In [18]:
# Use gpt-4o to optimize gpt-4o-mini's prompts

kwargs = dict(num_threads=THREADS, teacher_settings=dict(lm=gpt4o), prompt_model=gpt4o_mini)
optimizer = dspy.MIPROv2(metric=dataset.metric, auto="medium", **kwargs)

kwargs = dict(max_bootstrapped_demos=4, max_labeled_demos=4)
optimized_module = optimizer.compile(module, trainset=dataset.train, **kwargs)

2026/05/24 10:40:27 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING MEDIUM AUTO RUN SETTINGS:
num_trials: 18
minibatch: True
num_fewshot_candidates: 12
num_instruct_candidates: 6
valset size: 280

2026/05/24 10:40:27 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/05/24 10:40:27 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/05/24 10:40:27 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=12 sets of demonstrations...


Bootstrapping set 1/12
Bootstrapping set 2/12
Bootstrapping set 3/12


  7%|██████                                                                              | 5/70 [00:11<02:33,  2.35s/it]


Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
Bootstrapping set 4/12


  6%|████▊                                                                               | 4/70 [00:08<02:18,  2.09s/it]


...

2026/05/24 10:58:37 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 221.00 / 258 (85.7%):  94%|████████████████████████████████████████▏  | 262/280 [01:04<00:08,  2.22it/s]

2026/05/24 10:58:37 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 232.00 / 276 (84.1%): 100%|███████████████████████████████████████████| 280/280 [01:28<00:00,  3.18it/s]

2026/05/24 10:59:00 INFO dspy.evaluate.evaluate: Average Metric: 232.0 / 280 (82.9%)
2026/05/24 10:59:00 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [71.07, 82.86, 79.29, 87.86, 82.86]
2026/05/24 10:59:00 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 87.86
2026/05/24 10:59:00 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2026/05/24 10:59:00 INFO dspy.teleprompt.mipro_optimizer_v2: 

2026/05/24 10:59:00 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 87.86!


In [19]:
dspy.inspect_history()





[2026-05-24T10:59:00.367730]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## question ## ]]
{question}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "answer": "{answer}"
}
In adhering to this structure, your objective is: 
        You are a math tutor. Given the field `question`, provide a detailed reasoning process and the final answer in the fields `reasoning` and `answer`, respectively. Make sure to explain your thought process step-by-step to help the learner understand how to arrive at the solution.


User message:

[[ ## question ## ]]
There are two solutions for the equation $x^2 - x - 6 = 0$. What is the product of these two solutions?


Assistant message:

{
  "reasoning": "For a quadratic equation of th